# Lesson 3: Fix a bug in someone else's code

From [Forward Deployed Engineering](https://gradlabs.in/learn/fde/lessons/lesson3.html). The client, Northstar Skills, is made up: every name, number and message is invented.

You're the FDE at Northstar Skills. Kevin's code saves every WhatsApp enquiry to the lead sheet, but it saves the same person two or three times.

Fix it the way an FDE would: map the code, make the bug happen, write tests, fix it, and clean up the sheet.

For each cell: **guess what it will print first**, then press ▶ and check.

**1. Get Kevin's code.** This copies the course repo into Colab and moves into Kevin's project folder.

In [ ]:
import os
if not os.path.exists("/content/forward-deployed-engineering"):
    !git clone -q --depth 1 https://github.com/mo-ri-ch/forward-deployed-engineering /content/forward-deployed-engineering
%cd /content/forward-deployed-engineering/starter/northstar-leads

**2. Map it before you touch it.** Read the README first, then look at the files.

🤔 *Which file receives a WhatsApp message first?*

In [ ]:
!cat README.md

In [ ]:
!ls . tests

**3. Look at the lead sheet.** In production it's a Google Sheet. Here it's `leads.csv`, with the same columns.

In [ ]:
import pandas as pd

sheet = pd.read_csv("leads.csv", dtype=str).fillna("")
print(len(sheet), "rows")
sheet[["Enquiry date", "Name", "Phone", "Source"]].head(12)

**4. Follow one message.** Arun messages again. WhatsApp sends phone numbers as digits with the country code, like `919999901234`. We point the code at a copy of the sheet, so the real one stays safe.

🤔 *Arun is already in the sheet. Will `save_lead` say "added" or "updated"?*

In [ ]:
import shutil
shutil.copy("leads.csv", "/tmp/leads_copy.csv")
os.environ["LEADS_FILE"] = "/tmp/leads_copy.csv"   # experiment on a copy

import webhook, leads

payload = {"entry": [{"changes": [{"value": {
    "contacts": [{"profile": {"name": "Arun K"}}],
    "messages": [{"from": "919999901234", "text": {"body": "full stack weekend batch?"}}],
}}]}]}

msg = webhook.parse(payload)
print(msg)
print(leads.save_lead(msg))

It said **added**. Arun is now in the sheet four times. You've made the bug happen on purpose.

**5. Find every format that breaks.** A correct result is the last 10 digits. Run Kevin's `normalize_phone` on every number in the sheet, plus WhatsApp's format.

🤔 *How many formats break?*

In [ ]:
import re

def last10(raw):
    return re.sub(r"\D", "", raw)[-10:]

phones = list(sheet["Phone"].unique()) + ["919999901234"]
bad = [p for p in phones if leads.normalize_phone(p) != last10(p)]
print(len(bad), "formats break:")
for p in bad:
    print(f"  {p!r:20} → {leads.normalize_phone(p)!r}")

🤔 *How many different people are in the sheet? Kevin's code and the truth disagree.*

In [ ]:
print("Kevin's code counts", sheet["Phone"].map(leads.normalize_phone).nunique(), "people")
print("There are really   ", sheet["Phone"].map(last10).nunique(), "people")

**6. Run Kevin's tests first.** Before you change anything, see what passes today.

In [ ]:
!python -m pytest -q

All 3 pass. None of them uses a broken format, which is how the bug got through.

**7. Write tests that describe the bug.** One test for each broken format. They should fail now.

In [ ]:
%%writefile tests/test_phone_formats.py
from leads import normalize_phone


def test_whatsapp_sends_country_code_without_plus():
    assert normalize_phone("919999901234") == "9999901234"


def test_leading_zero():
    assert normalize_phone("099999 01236") == "9999901236"


def test_dashes():
    assert normalize_phone("99999-01241") == "9999901241"


def test_plus_91_with_dashes():
    assert normalize_phone("+91-99999-01234") == "9999901234"

In [ ]:
!python -m pytest -q

4 fail and 3 pass. Good: the failing tests are now your definition of "fixed".

**8. Your turn: write the fix.** Keep only the digits, then take the last 10.

Hints: `re.sub(r"\D", "", raw)` removes everything that isn't a digit. `text[-10:]` takes the last 10 characters.

In [ ]:
def my_normalize_phone(raw):
    # Write your version here
    ...


checks = [("919999901234", "9999901234"), ("099999 01236", "9999901236"), ("99999-01241", "9999901241"),
          ("+91-99999-01234", "9999901234"), ("+91 99999 01234", "9999901234"), ("9999901234", "9999901234"), ("", "")]
for raw, want in checks:
    got = my_normalize_phone(raw)
    print("✓" if got == want else "✗", repr(raw), "→", repr(got))

**9. Put the fix into Kevin's code.** Change `normalize_phone` and nothing else. This cell swaps in the fixed function. Compare it with yours, then look at the diff: it's exactly what Kevin will review.

In [ ]:
src = open("leads.py").read()
old = src[src.index("def normalize_phone"):src.index("def find_lead")]
new = r'''def normalize_phone(raw):
    digits = re.sub(r"\D", "", raw)   # keep only 0-9
    return digits[-10:]               # the last 10 digits


'''
src = src.replace(old, new)
if "import re" not in src:
    src = src.replace("from datetime import date", "import re\nfrom datetime import date")
open("leads.py", "w").write(src)

!git diff leads.py

**10. Run all the tests again.**

🤔 *How many pass now?*

In [ ]:
!python -m pytest -q

All 7 pass: your 4 new tests, and Kevin's 3 still work.

**11. Send Arun's message again,** on a fresh copy of the sheet. Python remembers the old code, so reload it first.

🤔 *"added" or "updated" this time?*

In [ ]:
import importlib

shutil.copy("leads.csv", "/tmp/leads_copy.csv")
importlib.reload(leads)
print(leads.save_lead(msg))

**12. Clean up the duplicates already in the sheet.** The fix stops new duplicates, but the old ones are still there. Find them first.

In [ ]:
sheet["Phone (10 digits)"] = sheet["Phone"].map(leads.normalize_phone)
dupes = sheet[sheet.duplicated("Phone (10 digits)", keep=False)].sort_values(["Phone (10 digits)", "Enquiry date"])
print(len(dupes), "rows belong to", dupes["Phone (10 digits)"].nunique(), "people")
dupes[["Enquiry date", "Name", "Phone", "Status", "Counsellor"]]

Now keep one row per person, and save it as a **new** file. Never overwrite the original.

In [ ]:
clean = sheet.sort_values("Enquiry date").drop_duplicates("Phone (10 digits)", keep="last")
clean.drop(columns="Phone (10 digits)").to_csv("leads_clean.csv", index=False)
print(len(sheet), "rows →", len(clean), "rows")
clean[clean["Name"].str.startswith("Rahul")][["Enquiry date", "Name", "Status"]]

Look at Rahul. Keeping the latest row says "follow up", but he has already **joined**. Keeping the latest row threw away the most important fact.

That's why a clean-up like this runs once, on a backup, with Kevin watching, and a counsellor checks the merged rows. 🤔 *Can you change the rule so a JOINED status always wins?*

**13. Write the pull request note.** Four short parts, in your own words:

- **Title:** what changes for the user, e.g. *Store phone numbers as 10 digits, so one person is one lead*
- **Why:** the real problem and its effect, with no blame
- **What changed:** exactly which function, and which tests you added
- **Risk:** what could go wrong, e.g. numbers from outside India lose their country code

**14. Put it on your GitHub.** This is your first portfolio piece.

1. Fork [the course repo](https://github.com/mo-ri-ch/forward-deployed-engineering) on GitHub.
2. In your fork, edit `starter/northstar-leads/leads.py` and add `tests/test_phone_formats.py`.
3. Open a pull request in your fork, with your note from step 13 as the description.

This cell downloads your two files, so you can paste them in.

In [ ]:
try:
    from google.colab import files
    files.download("leads.py")
    files.download("tests/test_phone_formats.py")
except ImportError:
    print("Not in Colab: your files are in", os.getcwd())